In [0]:
from pyspark.sql.functions import col, lit

dbutils.widgets.text("process_date", "")
process_date = dbutils.widgets.get("process_date")

df_api = (
    spark.table("chilecompra.bronze.ordenes_compra_api")
    .filter(
        col("_process_date") == lit(process_date).cast("date")
    )
)

In [0]:
estado_data = [
    (4, "Enviada a Proveedor"),
    (5, "En proceso"),
    (6, "Aceptada"),
    (7, "Solicitud de cancelación"),
    (9, "Cancelada"),
    (12, "Recepción Conforme"),
    (13, "Pendiente de Recepcionar"),
    (14, "Recepcionada Parcialmente"),
    (15, "Recepción Conforme Incompleta"),
]

df_estado_map = spark.createDataFrame(
    estado_data,
    ["codigo_estado", "estado"]
)

df_api = (
    df_api
    .join(
        df_estado_map,
        on="codigo_estado",
        how="left"
    )
)

In [0]:
total_rows = df_api.count()

null_codigo = (
    df_api
    .filter(col("codigo").isNull())
    .count()
)

duplicate_codigo = (
    total_rows
    - df_api.select("codigo").distinct().count()
)

unmapped_estado = (
    df_api
    .filter(col("estado").isNull())
    .count()
)

if total_rows == 0:
    raise ValueError(
        f"DQ FAILED: no OC rows for process_date={process_date}"
    )

if null_codigo > 0:
    raise ValueError(
        f"DQ FAILED: {null_codigo} rows have codigo NULL"
    )

if duplicate_codigo > 0:
    raise ValueError(
        f"DQ FAILED: {duplicate_codigo} duplicated codigo values"
    )

if unmapped_estado > 0:
    raise ValueError(
        f"DQ FAILED: {unmapped_estado} rows have unmapped estado"
    )

print(
    f"Pre-merge OC API DQ passed: {total_rows} orders"
)

In [0]:
from delta.tables import DeltaTable

target_table = "chilecompra.silver.ordenes_compra"

delta_target = DeltaTable.forName(
    spark,
    target_table
)

api_is_newer = """
    t._last_api_process_date IS NULL

    OR s._process_date > t._last_api_process_date

    OR (
        s._process_date = t._last_api_process_date
        AND (
            t._last_api_source_created_at IS NULL
            OR s._source_created_at > t._last_api_source_created_at
        )
    )
"""

In [0]:
(
    delta_target.alias("t")
    .merge(
        df_api.alias("s"),
        "t.codigo = s.codigo"
    )
    .whenMatchedUpdate(
        condition=api_is_newer,
        set={
            "nombre": "s.nombre",
            "codigo_estado": "s.codigo_estado",
            "estado": "s.estado",
            "_last_api_process_date": "s._process_date",
            "_last_api_source_created_at": "s._source_created_at"
        }
    )
    .whenNotMatchedInsert(
        values={
            "codigo": "s.codigo",
            "nombre": "s.nombre",
            "codigo_estado": "s.codigo_estado",
            "estado": "s.estado",
            "_last_api_process_date": "s._process_date",
            "_last_api_source_created_at": "s._source_created_at"
        }
    )
    .execute()
)

In [0]:
df_silver = spark.table(target_table)

missing_in_silver = (
    df_api
    .select("codigo")
    .join(
        df_silver.select("codigo"),
        on="codigo",
        how="left_anti"
    )
    .count()
)

if missing_in_silver > 0:
    raise ValueError(
        f"DQ FAILED: {missing_in_silver} current API orders "
        f"are missing from Silver"
    )

print(
    f"Silver incremental OC DQ passed: "
    f"all {total_rows} current API orders are present"
)